In [1]:
import os
import json
from pathlib import Path
from collections import Counter
import re # 정규표현식 모듈 임포트

# 1. 경로 설정 (원본 데이터셋 경로)
origin_root = Path("origin_sample_dataset") 

# 2. 클래스 매핑 정보 (이전 코드와 동일하게 유지)
class_mapping = {
    ("01", 0): 0, # '01' (배) + disease 0 = 0: '배 정상'
    ("01", 1): 1, # '01' (배) + disease 1 = 1: '배검은별무늬병'
    ("01", 2): 2, # '01' (배) + disease 2 = 2: '배과수화상병'

    ("02", 0): 8, # '02' (사과) + disease 0 = 8: '사과 정상'
    ("02", 3): 3, # '02' (사과) + disease 1 = 3: '사과갈색무늬병'
    ("02", 4): 4, # '02' (사과) + disease 2 = 4: '사과과수화상병'
    ("02", 5): 5, # '02' (사과) + disease 3 = 5: '사과부란병'
    ("02", 6): 6, # '02' (사과) + disease 4 = 6: '사과점무늬낙엽병'
    ("02", 7): 7  # '02' (사과) + disease 5 = 7: '사과탄저병'
}

# 클래스 ID와 이름 매핑 (결과 출력 시 가독성을 위해)
class_names = {
    0: '배 정상',
    1: '배검은별무늬병',
    2: '배과수화상병',
    3: '사과갈색무늬병',
    4: '사과과수화상병',
    5: '사과부란병',
    6: '사과점무늬낙엽병',
    7: '사과탄저병',
    8: '사과 정상'
}

def get_class_id_from_json(json_path: Path):
    """
    JSON 파일을 읽어 해당 파일 내의 모든 바운딩 박스에 대한 클래스 ID를 반환합니다.
    (바운딩 박스가 여러 개인 경우, 각 박스에 대해 동일한 클래스 ID를 반환)
    """
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        # print(f"  ! Error reading JSON file {json_path}: {e}")
        return []

    # 파일명에서 작물 정보 추출
    # 정규표현식을 사용하여 '_숫자숫자_' 패턴을 찾음
    match = re.search(r'_\d{2}_(\d{2})_', json_path.stem)
    if match and len(match.groups()) > 0:
        crop_code = match.group(1) # 예: '01' 또는 '02'
    else:
        # print(f"    ! Warning: Could not extract crop code from filename: {json_path.stem}. Skipping.")
        return []

    # JSON에서 질병 정보 추출
    try:
        json_disease_value = data['annotations']['disease']
    except KeyError:
        # print(f"    ! Error: Missing 'annotations.disease' key in JSON file {json_path.name}. Skipping.")
        return []

    # class_mapping을 사용하여 최종 class_id 결정
    class_key = (crop_code, json_disease_value)
    class_id = class_mapping.get(class_key)
    
    if class_id is None:
        # print(f"    ! Warning: No class mapping for {class_key} (from {json_path.name}). Skipping.")
        return []

    # JSON 내 'points' (바운딩 박스) 수만큼 클래스 ID를 반환 (각 바운딩 박스 = 1 인스턴스)
    # 'annotations' 또는 'points'가 없는 경우도 처리
    if 'annotations' not in data or 'points' not in data['annotations']:
        # print(f"    ! Warning: Missing 'annotations' or 'points' in JSON file {json_path.name}.")
        # 바운딩 박스가 없어도 클래스 정보는 있으므로, 이 경우 1개의 클래스 인스턴스로 간주
        # (만약 바운딩 박스가 없는 파일을 완전히 무시하고 싶다면 [] 반환)
        # 단, 모델 학습 시 바운딩 박스 없는 이미지는 라벨 파일이 비어있게 되므로, 실제로는 이런 파일은 학습에 사용되지 않음.
        # 여기서는 "해당 클래스의 이미지 수" 관점에서 1개로 카운트할게요.
        return [class_id] 

    # 실제 바운딩 박스 개수만큼 클래스 ID 반환
    # 이미지 크기를 검증해야 더 정확하지만, 여기서는 단순히 'points' 개수만 사용
    # 참고: convert_label 함수와 달리 바운딩 박스 유효성 검증(너비/높이 0)은 하지 않음.
    # 이는 단순히 "JSON 파일에 명시된 바운딩 박스 수"를 세기 위함.
    return [class_id] * len(data['annotations']['points'])

def analyze_dataset_distribution():
    """
    원본 데이터셋의 Training과 Validation 폴더를 스캔하여
    클래스 ID별 인스턴스 분포를 집계하고 출력합니다.
    """
    total_class_counts = Counter()

    print("--- Class Distribution Analysis ---")

    for dataset_type in ["Training", "Validation"]:
        print(f"\nScanning {dataset_type} folder...")
        dataset_path = origin_root / dataset_type
        
        if not dataset_path.exists():
            print(f"  Warning: {dataset_path} does not exist. Skipping.")
            continue

        current_type_counts = Counter()
        json_folders = [f for f in dataset_path.iterdir() if f.is_dir() and f.name.startswith("[라벨]")]
        
        if not json_folders:
            print(f"  No '[라벨]' folders found in {dataset_path}.")
            continue

        for folder in json_folders:
            json_files = list(folder.glob("*.json"))
            # print(f"  Found {len(json_files)} JSON files in {folder.name}")
            
            for json_file in json_files:
                class_ids_in_file = get_class_id_from_json(json_file)
                for class_id in class_ids_in_file:
                    current_type_counts[class_id] += 1
                    total_class_counts[class_id] += 1
        
        print(f"\n--- {dataset_type} Class Counts ---")
        if not current_type_counts:
            print("  No class instances found.")
        else:
            # 클래스 ID 순서대로 정렬하여 출력
            for class_id in sorted(current_type_counts.keys()):
                print(f"  - Class {class_id} ({class_names.get(class_id, 'Unknown')}): {current_type_counts[class_id]} instances")
        print("---------------------------------")

    print("\n--- Total Class Counts Across All Datasets ---")
    if not total_class_counts:
        print("No class instances found in total.")
    else:
        # 전체 클래스 ID 순서대로 정렬하여 출력
        for class_id in sorted(total_class_counts.keys()):
            print(f"  - Class {class_id} ({class_names.get(class_id, 'Unknown')}): {total_class_counts[class_id]} instances")
    print("---------------------------------------------")
    print("\nAnalysis complete.")

if __name__ == "__main__":
    analyze_dataset_distribution()

--- Class Distribution Analysis ---

Scanning Training folder...

Scanning Validation folder...

--- Total Class Counts Across All Datasets ---
No class instances found in total.
---------------------------------------------

Analysis complete.
